In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("fermi.csv")

fig = px.scatter(
    df, 
    x="t90", 
    y="fluence", 
    color="Classified", 
    labels={"t90": "T90 (s)", "fluence": "Fluence (erg/cm²)", "Classified": "GRB Class"}, 
    title="Fermi/GBM",
    template="plotly_white"
)

fig.update_traces(marker=dict(size=8, opacity=0.7, line=dict(width=0.5, color="black")))

fig.update_layout(
    xaxis=dict(title="T90 (s)", gridcolor="lightgrey", type="log"),
    yaxis=dict(title="Fluence (erg/cm²)", gridcolor="lightgrey", type="log"),
    coloraxis_colorbar=dict(title="GRB Class")
)

fig.show()





In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go 

df = pd.read_csv("fermi.csv")

fig.add_trace(go.Scatter(
    x=df["t50"],
    y=df["t90"],

    mode='markers',
    marker=dict(
        size=6,  
        color=df["Classified"],  
        colorscale="Viridis",   # Gradient color scale
        opacity=0.8,
        line=dict(width=0.5, color='black')  # Thin border around markers
    ),
    name='GRB Data'
))

fig.update_layout(
    
    xaxis=dict(
        title="T50 (s)",
        gridcolor='rgba(200, 200, 200, 0.3)',
        zerolinecolor='rgba(0, 0, 0, 0.5)',
        type="log"  # Logarithmic scale for x-axis
    ),
    yaxis=dict(
        title="T90 (s)",
        gridcolor='rgba(200, 200, 200, 0.3)',
        zerolinecolor='rgba(0, 0, 0, 0.5)',
        type="log"  # Logarithmic scale for y-axis
    ),
    template="plotly_white",
    plot_bgcolor='rgba(240, 240, 240, 0.8)',
    legend=dict(title="Legend", font=dict(size=10)),
    font=dict(family="Arial, sans-serif", size=12)
)

fig.show()




In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import plotly.express as px
from astroquery.ned import Ned
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

df = pd.read_csv("fermi.csv")

if "Undefined_Cluster" in df.columns:
    for cluster in sorted(df["Undefined_Cluster"].dropna().unique()):  
        print(f"Analysis for New Cluster {int(cluster)}:")
        cluster_data = df[df["Undefined_Cluster"] == cluster]
        print(cluster_data.describe())
else:
    print("No undefined clusters to analyze.")

numerical_columns = [
    "t90", "t50", "fluence" , "flnc_band_beta", "duration_energy_low",
    "flux_1024", "flux_64", "flnc_band_ampl",
    "flnc_band_epeak", "flnc_band_alpha", "duration_energy_high", "flu_low", "flu_high", "flux_256"
]

df = df.dropna(subset=numerical_columns)

def classify_grb_binary(t90):
    return "Short" if t90 <= 2 else "Long"

df["Classified"] = df["t90"].apply(classify_grb_binary)

def assign_progenitor(grb_class):
    if grb_class == "Short":
        return "Type I (Mergers: NS-NS or NS-BH)"
    elif grb_class == "Long":
        return "Type II (Collapsing Massive Stars)"
    return "Unclassified"

df["Progenitor_Type"] = df["Classified"].apply(assign_progenitor)

class_mapping = {"Short": 0, "Long": 1}
df["grb_class_label"] = df["Classified"].map(class_mapping)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numerical_columns])

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, df["grb_class_label"], test_size=0.2, random_state=42
)

clf = SVC(kernel="rbf", random_state=42, probability=True)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")
print(f"Confusion Matrix:\n{conf_matrix}")

clustering = DBSCAN(eps=2, min_samples=5).fit(X_scaled)
df["Cluster"] = clustering.labels_

def refine_progenitor(row):
    if row["Cluster"] == -1:
        return "Unusual Cluster"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor, axis=1)

undefined_data = df[df["Cluster"] == -1][numerical_columns]
kmeans = KMeans(n_clusters=2, random_state=42)
undefined_clusters = kmeans.fit_predict(undefined_data)
df.loc[df["Cluster"] == -1, "Undefined_Cluster"] = undefined_clusters

def refine_progenitor_with_patterns(row):
    if row["Cluster"] == -1:
        return f"New Cluster {int(row['Undefined_Cluster'])}"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor_with_patterns, axis=1)

pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)

df["pca_1"] = pca_results[:, 0]
df["pca_2"] = pca_results[:, 1]

color_map = {
    "Type I (Mergers: NS-NS or NS-BH)": "blue",
    "Type II (Collapsing Massive Stars)": "green",
    "Unusual Cluster": "red",
    "New Cluster 0": "orange",
    "New Cluster 1": "purple",
}

fig_pca = px.scatter(
    df,
    x="pca_1",
    y="pca_2",
    color="Refined_Progenitor_Type",
    title="PCA Visualization with Refined Progenitor Types and Clusters",
    labels={"pca_1": "PCA Dimension 1", "pca_2": "PCA Dimension 2"},
    color_discrete_map=color_map,
    template="plotly"
)
fig_pca.show()

if "Undefined_Cluster" in df.columns:
    for cluster in sorted(df["Undefined_Cluster"].dropna().unique()):
        print(f"Analysis for New Cluster {int(cluster)}:")
        cluster_data = df[df["Undefined_Cluster"] == cluster]
        print(cluster_data.describe())
else:
    print("No undefined clusters to analyze.")

C:\Users\elife\AppData\Local\Temp\ipykernel_13176\2703224386.py:10: DeprecationWarning:

the ``ned`` module has been moved to astroquery.ipac.ned, please update your imports.



No undefined clusters to analyze.
Accuracy: 0.9414225941422594
Recall: 0.9526143790849673
Precision: 0.9781879194630873
Confusion Matrix:
[[ 92  13]
 [ 29 583]]


Analysis for New Cluster 0:
              t90  t90_error   t90_start       fluence  fluence_error  \
count   40.000000  40.000000   40.000000  4.000000e+01   4.000000e+01   
mean   199.471325   2.970025  -21.977575  2.585580e-04   6.329923e-07   
std    221.203936   4.390635  138.758916  6.235320e-04   2.262785e-06   
min      0.032000   0.023000 -807.424000  1.545500e-07   2.073500e-08   
25%      5.216000   0.545500   -1.664000  1.340525e-05   4.974300e-08   
50%     88.705000   1.556000   -0.008000  3.475600e-05   1.442600e-07   
75%    376.326000   3.345250    2.372000  1.988125e-04   3.176250e-07   
max    828.672000  23.752000  188.451000  3.147500e-03   1.437300e-05   

         flux_1024  flux_1024_error  flux_1024_time      flux_64  \
count    40.000000        40.000000       40.000000    40.000000   
mean    159.396388        11.264525       51.144300   339.365090   
std     234.900530        45.298170      106.689985   554.308325   
min       0.945500         0.180100     -1

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from sklearn.decomposition import PCA
import plotly.express as px
from lime.lime_tabular import LimeTabularExplainer

# Load dataset
df = pd.read_csv("fermi.csv")

# Define numerical columns
numerical_columns = [
    "t90", "t50", "fluence", "flnc_band_beta", "duration_energy_low",
    "flux_1024", "flux_64", "flnc_band_ampl",
    "flnc_band_epeak", "flnc_band_alpha", "duration_energy_high", "flu_low", "flu_high", "flux_256"
]

# Ensure there are no NaN values in the numerical columns
df = df.dropna(subset=numerical_columns)

# Binary classification for GRBs
def classify_grb_binary(t90):
    return "Short" if t90 <= 2 else "Long"

df["Classified"] = df["t90"].apply(classify_grb_binary)

# Encode the target variable
class_mapping = {"Short": 0, "Long": 1}
df["grb_class_label"] = df["Classified"].map(class_mapping)

# Standardize numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numerical_columns])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, df["grb_class_label"], test_size=0.2, random_state=42
)

# SVM model training
clf = SVC(kernel="rbf", random_state=42, probability=True)
clf.fit(X_train, y_train)

# Model evaluation
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")
print(f"Confusion Matrix:\n{conf_matrix}")

# Using LIME for local explanations
explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=numerical_columns,
    class_names=["Short", "Long"],
    mode="classification"
)

# Select a sample from the test set for explanation
sample_index = 0
sample = X_test[sample_index].reshape(1, -1)
sample_prediction = clf.predict_proba(sample)[0]

print(f"Sample Prediction (Class Probabilities): {sample_prediction}")

# Explain the prediction
lime_exp = explainer.explain_instance(
    data_row=X_test[sample_index],
    predict_fn=clf.predict_proba
)

# Display explanation
lime_exp.show_in_notebook()

# PCA for visualization
pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)

df["pca_1"] = pca_results[:, 0]
df["pca_2"] = pca_results[:, 1]

# Color map for classifications
color_map = {
    0: "blue",
    1: "green"
}

# Visualize PCA results
fig_pca = px.scatter(
    df,
    x="pca_1",
    y="pca_2",
    color="grb_class_label",
    title="PCA Visualization with GRB Classes",
    labels={"pca_1": "PCA Dimension 1", "pca_2": "PCA Dimension 2"},
    color_discrete_map=color_map,
    template="plotly"
)
fig_pca.show()



Accuracy: 0.9414225941422594
Recall: 0.9526143790849673
Precision: 0.9781879194630873
Confusion Matrix:
[[ 92  13]
 [ 29 583]]
Sample Prediction (Class Probabilities): [0.81690098 0.18309902]


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import plotly.express as px
from astroquery.ned import Ned
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

df = pd.read_csv("fermi.csv")

if "Undefined_Cluster" in df.columns:
    for cluster in sorted(df["Undefined_Cluster"].dropna().unique()):  
        print(f"Analysis for New Cluster {int(cluster)}:")
        cluster_data = df[df["Undefined_Cluster"] == cluster]
        print(cluster_data.describe())
else:
    print("No undefined clusters to analyze.")



def classify_grb_binary(t90):
    return "Short" if t90 <= 2 else "Long"

df["Classified"] = df["T90"].apply(classify_grb_binary)

def assign_progenitor(grb_class):
    if grb_class == "Short":
        return "Type I (Mergers: NS-NS or NS-BH)"
    elif grb_class == "Long":
        return "Type II (Collapsing Massive Stars)"
    else:
        return "Unclassified"

df["Progenitor_Type"] = df["Classified"].apply(assign_progenitor)

class_mapping = {"Short": 0, "Long": 1}
df["grb_class_label"] = df["Classified"].map(class_mapping)

numerical_columns = [
    "t90", "t50", "fluence" , "flnc_band_beta", "duration_energy_low",
    "flux_1024", "flux_64", "flnc_band_ampl",
    "flnc_band_epeak", "flnc_band_alpha", "duration_energy_high", "flu_low", "flu_high", "flux_256"
]

df = df.dropna(subset=numerical_columns)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numerical_columns])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, df["grb_class_label"], test_size=0.2, random_state=42
)

# Train the SVM model
clf = SVC(kernel="rbf", random_state=42, probability=True)
clf.fit(X_train, y_train)

# Evaluate the model
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average="binary")
precision = precision_score(y_test, y_pred, average="binary")
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")
print(f"Confusion Matrix:\n{conf_matrix}")

# Count the number of Short and Long GRBs before clustering
print("Before Machine Learning Clustering:")
print(df["Classified"].value_counts())

# Perform clustering using DBSCAN
clustering = DBSCAN(eps=2, min_samples=5).fit(X_scaled)
df["Cluster"] = clustering.labels_

# Refine progenitor types based on cluster labels
def refine_progenitor(row):
    if row["Cluster"] == -1:
        return "Unusual Cluster"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor, axis=1)

# Further refine undefined clusters using KMeans
undefined_data = df[df["Cluster"] == -1][numerical_columns]
kmeans = KMeans(n_clusters=2, random_state=42)
undefined_clusters = kmeans.fit_predict(undefined_data)
df.loc[df["Cluster"] == -1, "Undefined_Cluster"] = undefined_clusters

def refine_progenitor_with_patterns(row):
    if row["Cluster"] == -1:
        return f"New Cluster {int(row['Undefined_Cluster'])}"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor_with_patterns, axis=1)

# Count the number of Short and Long GRBs after clustering
print("After Machine Learning Clustering:")
print(df["Refined_Progenitor_Type"].value_counts())

# PCA for visualization
pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)
df["pca_1"] = pca_results[:, 0]
df["pca_2"] = pca_results[:, 1]

# Plot PCA results
fig_pca = px.scatter(
    df,
    x="pca_1",
    y="pca_2",
    color="Refined_Progenitor_Type",
    title="PCA Visualization with Progenitor Types and Clusters",
    labels={"pca_1": "PCA Dimension 1", "pca_2": "PCA Dimension 2"},
    template="plotly"
)
fig_pca.show()

# Plot RA and Dec after clustering
fig_ra_dec = px.scatter(
    df,
    x="RA",
    y="Dec",
    color="Refined_Progenitor_Type",
    title="RA and Dec of GRBs with Clusters",
    labels={"RA": "Right Ascension (RA)", "Dec": "Declination (Dec)"},
    template="plotly"
)
fig_ra_dec.show()

# Analyze new clusters
if "Undefined_Cluster" in df.columns:
    for cluster in sorted(df["Undefined_Cluster"].dropna().unique()):
        print(f"Analysis for New Cluster {int(cluster)}:")
        cluster_data = df[df["Undefined_Cluster"] == cluster]
        print(cluster_data.describe())
else:
    print("No undefined clusters to analyze.")

No undefined clusters to analyze.


KeyError: 'T90'